# 02 — Data Preprocessing (Updated)




In [1]:
from pathlib import Path
import sys
import html
import json
import re

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Resolve project root robustly
# ------------------------------------------------------------

cwd = Path.cwd()

candidates = [
    cwd,
    cwd.parent,
]

PROJECT_ROOT = None

for candidate in candidates:
    if (candidate / "data" / "raw").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    # Fallback for the usual project structure where this notebook
    # is stored inside notebooks/
    PROJECT_ROOT = cwd.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "pharmacy_review.csv"
OUTPUT_PATH = PROCESSED_DIR / "cleaned_reviews.csv"
LABEL_MAPPING_PATH = ARTIFACTS_DIR / "label_mapping.json"
CONFLICT_REPORT_PATH = REPORTS_DIR / "conflicting_review_labels.csv"
NER_OUTPUT_PATH = PROCESSED_DIR / "ner_enriched_reviews.csv"

print("Project root:", PROJECT_ROOT)
print("Input dataset:", DATA_PATH)


Project root: c:\Users\yashika\OneDrive\Pictures\medicine_review_classification_updated
Input dataset: c:\Users\yashika\OneDrive\Pictures\medicine_review_classification_updated\data\raw\pharmacy_review.csv


In [2]:
# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}\n"
        "Update DATA_PATH if your raw dataset uses a different filename."
    )

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)
display(df.head())


Original shape: (24000, 7)


,id,gender,age,medicine,condition,review,label
0,0,Male,76.0,Glimepiride,Type 2 Diabetes,I am still waiting to see if the intended bene...,neutral
1,1,Female,44.0,Sertraline,Depression,My calendar has nausea beginning on the twenty...,side effect
2,2,Female,58.0,Diclofenac,Pain,I am keeping track of how Diclofenac is going ...,no side effect
3,3,Female,31.0,Carbimazole,Hyperthyroidism,This is an update from my Carbimazole treatmen...,side effect
4,4,Male,41.0,Escitalopram,Depression,Started a course of Escitalopram for depressio...,side effect


In [3]:
# ------------------------------------------------------------
# Basic dataset inspection
# ------------------------------------------------------------

print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

if "label" in df.columns:
    print("\nRaw label distribution:")
    display(df["label"].value_counts(dropna=False))


Columns:
['id', 'gender', 'age', 'medicine', 'condition', 'review', 'label']

Missing values:


condition    1927
age          1520
gender       1307
medicine     1243
id              0
review          0
label           0
dtype: int64


Raw label distribution:


label
neutral           8000
side effect       8000
no side effect    8000
Name: count, dtype: int64

## 1. Keep only rows required for NLP training

The classifier is trained from review text and label. Therefore, rows should **not** be deleted just because optional metadata such as `age`, `gender`, `medicine`, or `condition` is missing.


In [4]:
# ------------------------------------------------------------
# Validate required columns
# ------------------------------------------------------------

required_columns = {"review", "label"}

missing_required = required_columns - set(df.columns)

if missing_required:
    raise KeyError(
        f"Missing required columns: {sorted(missing_required)}"
    )

# Keep rows only when review and label are available
df = df.dropna(
    subset=["review", "label"]
).copy()

print("Shape after removing missing review/label:", df.shape)


Shape after removing missing review/label: (24000, 7)


## 2. Optional metadata cleanup

These fields are retained for analysis, but they are **not** required for the text classifier. Missing values in these columns are not used to remove training reviews.


In [5]:
# ------------------------------------------------------------
# Optional metadata cleanup
# ------------------------------------------------------------

if "age" in df.columns:
    df["age"] = pd.to_numeric(
        df["age"],
        errors="coerce"
    )

    invalid_age = (
        (df["age"] < 12)
        | (df["age"] > 88)
    )

    df.loc[invalid_age, "age"] = np.nan

print("Rows retained:", len(df))


Rows retained: 24000


## 3. Text cleaning

The cleaning is intentionally light because TF-IDF can work well with natural text.

### We keep:
- `no`
- `not`
- `never`
- `without`
- symptom names
- useful context words such as `but`, `however`, `later`

### We do not apply:
- aggressive stemming
- generic stopword removal
- punctuation stripping that could unnecessarily alter sentence structure


In [6]:
# ------------------------------------------------------------
# Shared preprocessing implementation
# ------------------------------------------------------------

from src.preprocessing import (
    EMOJI_MAP,
    clean_review,
    expand_negative_contractions,
)

print("Loaded shared preprocessing from src.preprocessing")


Loaded shared preprocessing from src.preprocessing


In [7]:
# ------------------------------------------------------------
# Quick cleaning tests
# ------------------------------------------------------------

test_reviews = [
    "I didn't experience nausea.",
    "I had no headache. Later I developed nausea.",
    "<br>Metformin helped me 😊",
    "I wasn't dizzy, but I had vomiting 🤮",
]

for sample in test_reviews:
    print("ORIGINAL:", sample)
    print("CLEANED :", clean_review(sample))
    print("-" * 70)


ORIGINAL: I didn't experience nausea.
CLEANED : i did not experience nausea.
----------------------------------------------------------------------
ORIGINAL: I had no headache. Later I developed nausea.
CLEANED : i had no headache. later i developed nausea.
----------------------------------------------------------------------
ORIGINAL: <br>Metformin helped me 😊
CLEANED : metformin helped me happy
----------------------------------------------------------------------
ORIGINAL: I wasn't dizzy, but I had vomiting 🤮
CLEANED : i was not dizzy, but i had vomiting vomit
----------------------------------------------------------------------


In [8]:
# ------------------------------------------------------------
# Apply text cleaning
# ------------------------------------------------------------

df["patient_review"] = (
    df["review"]
    .astype(str)
    .apply(clean_review)
)

# Remove reviews that became empty after cleaning
df = df[
    df["patient_review"]
    .str.len()
    .gt(0)
].copy()

print("Shape after text cleaning:", df.shape)

display(
    df[["review", "patient_review", "label"]]
    .head()
)


Shape after text cleaning: (24000, 8)


,review,patient_review,label
0,I am still waiting to see if the intended bene...,i am still waiting to see if the intended bene...,neutral
1,My calendar has nausea beginning on the twenty...,my calendar has nausea beginning on the twenty...,side effect
2,I am keeping track of how Diclofenac is going ...,i am keeping track of how diclofenac is going ...,no side effect
3,This is an update from my Carbimazole treatmen...,this is an update from my carbimazole treatmen...,side effect
4,Started a course of Escitalopram for depressio...,started a course of escitalopram for depressio...,side effect


## 4. Normalize labels

Labels are converted to lowercase and stripped before encoding. The mapping is sorted so it does not depend on dataset row order.


In [9]:
# ------------------------------------------------------------
# Normalize labels
# ------------------------------------------------------------

df["label_text"] = (
    df["label"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Normalized labels:")
print(sorted(df["label_text"].unique()))


Normalized labels:
['neutral', 'no side effect', 'side effect']


## 5. Check duplicate reviews with conflicting labels

Before removing duplicate reviews, check whether the **same cleaned review** appears with different labels.

If conflicting duplicates exist, they are saved to a report and removed from the training dataset rather than arbitrarily keeping one label.


In [10]:
# ------------------------------------------------------------
# Find conflicting duplicate labels
# ------------------------------------------------------------

label_counts_per_review = (
    df.groupby("patient_review")["label_text"]
    .nunique()
)

conflicting_review_texts = (
    label_counts_per_review[
        label_counts_per_review > 1
    ]
    .index
)

conflicting_rows = df[
    df["patient_review"].isin(
        conflicting_review_texts
    )
].copy()

print("Conflicting unique review texts:", len(conflicting_review_texts))
print("Rows involved in conflicts:", len(conflicting_rows))

if not conflicting_rows.empty:
    conflicting_rows.to_csv(
        CONFLICT_REPORT_PATH,
        index=False,
    )

    print(
        "Saved conflict report to:",
        CONFLICT_REPORT_PATH,
    )

    display(
        conflicting_rows[
            ["patient_review", "label_text"]
        ]
        .sort_values("patient_review")
        .head(20)
    )


Conflicting unique review texts: 0
Rows involved in conflicts: 0


In [11]:
# ------------------------------------------------------------
# Remove conflicting duplicate reviews
# ------------------------------------------------------------

if len(conflicting_review_texts) > 0:
    df = df[
        ~df["patient_review"].isin(
            conflicting_review_texts
        )
    ].copy()

print("Shape after removing conflicting reviews:", df.shape)


Shape after removing conflicting reviews: (24000, 9)


In [12]:
# ------------------------------------------------------------
# Remove exact duplicate cleaned reviews
# ------------------------------------------------------------

before_dedup = len(df)

df = (
    df.drop_duplicates(
        subset=["patient_review"],
        keep="first",
    )
    .copy()
)

removed_duplicates = (
    before_dedup - len(df)
)

print("Duplicate reviews removed:", removed_duplicates)
print("Shape after deduplication:", df.shape)


Duplicate reviews removed: 0
Shape after deduplication: (24000, 9)


## 6. Create deterministic label mapping


In [13]:
# ------------------------------------------------------------
# Label mapping
# ------------------------------------------------------------

labels = sorted(
    df["label_text"].unique()
)

label_dict = {
    label: index
    for index, label in enumerate(labels)
}

print("Label mapping:")
print(label_dict)

df["label"] = (
    df["label_text"]
    .map(label_dict)
    .astype(int)
)


Label mapping:
{'neutral': 0, 'no side effect': 1, 'side effect': 2}


## 7. Final validation

This section checks:
- missing cleaned text
- missing labels
- class distribution
- negation preservation


In [14]:
# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

print("Missing patient_review:", df["patient_review"].isna().sum())
print("Empty patient_review:", (df["patient_review"].str.len() == 0).sum())
print("Missing encoded label:", df["label"].isna().sum())

print("\nEncoded class distribution:")
display(
    df["label"]
    .value_counts()
    .sort_index()
)

print("\nClass percentage:")
display(
    df["label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)


Missing patient_review: 0
Empty patient_review: 0
Missing encoded label: 0

Encoded class distribution:


label
0    8000
1    8000
2    8000
Name: count, dtype: int64


Class percentage:


label
0    33.33
1    33.33
2    33.33
Name: proportion, dtype: float64

In [15]:
# ------------------------------------------------------------
# Verify important negation words were preserved
# ------------------------------------------------------------

NEGATION_TERMS = [
    " no ",
    " not ",
    " never ",
    " without ",
]

for term in NEGATION_TERMS:
    count = (
        df["patient_review"]
        .str.contains(
            re.escape(term.strip()),
            regex=True,
            na=False,
        )
        .sum()
    )

    print(
        f"{term.strip():>8}: {count} reviews"
    )


      no: 21857 reviews
     not: 18060 reviews
   never: 875 reviews
 without: 416 reviews


## 8. Structured NER extraction

NER is applied to the **original `review`** column so every extracted entity can preserve exact source evidence spans.

The shared project NER pipeline extracts:
- medications
- medication dosage, route, and frequency
- symptoms
- dates and durations
- negation, uncertainty, historical/current context, and subject attribution for symptoms
- medication–symptom **reported associations** (not clinical causation)
- the four-way narrative classification

The NER output is saved separately as `data/processed/ner_enriched_reviews.csv` so TF-IDF/model notebooks can continue using the compact `cleaned_reviews.csv`.


In [16]:
# ------------------------------------------------------------
# Run shared project NER extraction
# ------------------------------------------------------------

from src.dataset_ner import NER_COLUMNS, enrich_dataframe_with_ner

df_ner = enrich_dataframe_with_ner(
    df,
    source_column="review",
    continue_on_error=True,
    progress_every=1000,
)

ner_error_count = df_ner["ner_error"].astype(str).str.len().gt(0).sum()

print("NER rows:", len(df_ner))
print("NER row errors:", ner_error_count)

display(
    df_ner[[
        "review",
        "ner_medication_names",
        "ner_symptom_names",
        "ner_date_values",
        "ner_duration_values",
        "narrative_classification",
    ]].head()
)


NER processed: 1,000/24,000
NER processed: 2,000/24,000
NER processed: 3,000/24,000
NER processed: 4,000/24,000
NER processed: 5,000/24,000
NER processed: 6,000/24,000
NER processed: 7,000/24,000
NER processed: 8,000/24,000
NER processed: 9,000/24,000
NER processed: 10,000/24,000
NER processed: 11,000/24,000
NER processed: 12,000/24,000
NER processed: 13,000/24,000
NER processed: 14,000/24,000
NER processed: 15,000/24,000
NER processed: 16,000/24,000
NER processed: 17,000/24,000
NER processed: 18,000/24,000
NER processed: 19,000/24,000
NER processed: 20,000/24,000
NER processed: 21,000/24,000
NER processed: 22,000/24,000
NER processed: 23,000/24,000
NER processed: 24,000/24,000
NER rows: 24000
NER row errors: 0


,review,ner_medication_names,ner_symptom_names,ner_date_values,ner_duration_values,narrative_classification
0,I am still waiting to see if the intended bene...,"[""Glimepiride""]",[],[],[],No Side Effect
1,My calendar has nausea beginning on the twenty...,"[""Sertraline""]",[],[],[],Side Effect
2,I am keeping track of how Diclofenac is going ...,"[""Diclofenac""]",[],[],[],Insufficient Information
3,This is an update from my Carbimazole treatmen...,[],[],[],[],Insufficient Information
4,Started a course of Escitalopram for depressio...,"[""Escitalopram""]",[],[],[],Mixed Evidence


In [17]:
# ------------------------------------------------------------
# Inspect one full structured NER record
# ------------------------------------------------------------

if len(df_ner) > 0:
    sample = df_ner.iloc[0]
    print("Source review:")
    print(sample["review"])
    print("\nEntities with source spans:")
    print(json.dumps(json.loads(sample["ner_entities_json"]), indent=2, ensure_ascii=False))
    print("\nMedication–symptom reported associations:")
    print(json.dumps(json.loads(sample["ner_relations_json"]), indent=2, ensure_ascii=False))


Source review:
I am still waiting to see if the intended benefit lasts. Quick note about Glimepiride for type 2 diabetes: I did not experience nausea myself; my questions about Glimepiride come from hearing a my mother describe it.

Entities with source spans:
[
  {
    "id": "medication_1",
    "type": "MEDICATION",
    "value": "Glimepiride",
    "normalized": "Glimepiride",
    "span": {
      "text": "Glimepiride",
      "char_start": 74,
      "char_end": 85,
      "token_start": 14,
      "token_end": 15
    },
    "attributes": {}
  },
  {
    "id": "symptom_1",
    "type": "SYMPTOM",
    "value": "nausea",
    "normalized": "nausea",
    "span": {
      "text": "nausea",
      "char_start": 128,
      "char_end": 134,
      "token_start": 23,
      "token_end": 24
    },
    "assertion": {
      "negated": true,
      "uncertain": false,
      "temporal": "Current",
      "subject": "Patient"
    }
  },
  {
    "id": "medication_2",
    "type": "MEDICATION",
    "value": "Glime

## 9. Select final columns

The metadata is retained if it exists. The classifier can continue using only `patient_review` and `label`.


In [18]:
# ------------------------------------------------------------
# Final columns
# ------------------------------------------------------------

# Compact classifier dataset: unchanged downstream contract.
preferred_columns = [
    "id",
    "gender",
    "age",
    "medicine",
    "condition",
    "patient_review",
    "label",
]

final_columns = [
    column
    for column in preferred_columns
    if column in df.columns
]

df_final = (
    df[final_columns]
    .reset_index(drop=True)
)

# NER-enriched dataset: keeps original text because evidence offsets
# point into the unmodified source narrative.
ner_base_columns = [
    "id",
    "gender",
    "age",
    "medicine",
    "condition",
    "review",
    "patient_review",
    "label",
]

ner_final_columns = [
    column
    for column in ner_base_columns
    if column in df_ner.columns
] + list(NER_COLUMNS)

df_ner_final = (
    df_ner[ner_final_columns]
    .reset_index(drop=True)
)

print("Classifier dataset shape:", df_final.shape)
print("NER-enriched dataset shape:", df_ner_final.shape)

display(df_final.head())
display(
    df_ner_final[[
        column
        for column in [
            "review",
            "ner_medication_names",
            "ner_symptom_names",
            "narrative_classification",
        ]
        if column in df_ner_final.columns
    ]].head()
)


Classifier dataset shape: (24000, 7)
NER-enriched dataset shape: (24000, 18)


,id,gender,age,medicine,condition,patient_review,label
0,0,Male,76.0,Glimepiride,Type 2 Diabetes,i am still waiting to see if the intended bene...,0
1,1,Female,44.0,Sertraline,Depression,my calendar has nausea beginning on the twenty...,2
2,2,Female,58.0,Diclofenac,Pain,i am keeping track of how diclofenac is going ...,1
3,3,Female,31.0,Carbimazole,Hyperthyroidism,this is an update from my carbimazole treatmen...,2
4,4,Male,41.0,Escitalopram,Depression,started a course of escitalopram for depressio...,2


,review,ner_medication_names,ner_symptom_names,narrative_classification
0,I am still waiting to see if the intended bene...,"[""Glimepiride""]",[],No Side Effect
1,My calendar has nausea beginning on the twenty...,"[""Sertraline""]",[],Side Effect
2,I am keeping track of how Diclofenac is going ...,"[""Diclofenac""]",[],Insufficient Information
3,This is an update from my Carbimazole treatmen...,[],[],Insufficient Information
4,Started a course of Escitalopram for depressio...,"[""Escitalopram""]",[],Mixed Evidence


## 10. Save cleaned dataset, NER dataset, and label mapping


In [19]:
# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

df_final.to_csv(
    OUTPUT_PATH,
    index=False,
)

df_ner_final.to_csv(
    NER_OUTPUT_PATH,
    index=False,
)

with open(
    LABEL_MAPPING_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        label_dict,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Saved classifier dataset:")
print(OUTPUT_PATH)

print("\nSaved NER-enriched dataset:")
print(NER_OUTPUT_PATH)

print("\nSaved label mapping:")
print(LABEL_MAPPING_PATH)

print("\nFinal number of rows:")
print(len(df_final))


Saved classifier dataset:
c:\Users\yashika\OneDrive\Pictures\medicine_review_classification_updated\data\processed\cleaned_reviews.csv

Saved NER-enriched dataset:
c:\Users\yashika\OneDrive\Pictures\medicine_review_classification_updated\data\processed\ner_enriched_reviews.csv

Saved label mapping:
c:\Users\yashika\OneDrive\Pictures\medicine_review_classification_updated\artifacts\label_mapping.json

Final number of rows:
24000


## 11. Important next step

This notebook now produces **two aligned outputs**:

```text
02_data_preprocessing_updated.ipynb
        │
        ├── cleaned_reviews.csv
        │       ↓
        │   TF-IDF feature engineering
        │       ↓
        │   model experiments / evaluation
        │
        └── ner_enriched_reviews.csv
                ↓
            structured entity analysis
            evidence-span inspection
            assertion / relation analysis
```

The classifier should continue using `patient_review` from `cleaned_reviews.csv`.

The NER dataset uses the original `review` text for extraction because its character/token offsets must remain anchored to the source narrative.

Because preprocessing has changed, retrain TF-IDF and the classifier. Do **not** continue using an older `tfidf.joblib` or `best_model.joblib` that was trained using a different preprocessing pipeline.
